# Claude Agent SDK + LightRAG — Egyptian Law Ingestion

This notebook demonstrates how to use the **Claude Agent SDK** (the successor to the deprecated `claude-code-sdk`) for chat-style LLM calls, and how to wire it into **LightRAG** as the LLM backend for a knowledge-graph RAG pipeline over the Egyptian Law corpus.

This RAG will later be exposed as a **tool** to a multi-agent legal-explainer system.

## Auth

Unlike the older `claude-code-sdk` (which routes through the local Claude Code CLI), `claude-agent-sdk` authenticates directly with **`ANTHROPIC_API_KEY`** (or `ANTHROPIC_AUTH_TOKEN` + `ANTHROPIC_BASE_URL` for custom / proxy endpoints).

## Prerequisites

```bash
pip install claude-agent-sdk lightrag-hku pymupdf httpx python-dotenv numpy
```

And an Ollama instance running locally with a Qwen3 embedding model pulled:

```bash
ollama pull qwen3-embedding:0.6b
```

---
## 1. Imports & Auth

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from claude_agent_sdk import query, ClaudeAgentOptions
from claude_agent_sdk.types import AssistantMessage, TextBlock, ResultMessage

# Load API token + base URL (and any other secrets) from project .env
PROJECT_ROOT = Path("/Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM")
load_dotenv(PROJECT_ROOT / ".env")

# These three are forwarded to every SDK call via ClaudeAgentOptions(env=...).
# Use ANTHROPIC_AUTH_TOKEN + ANTHROPIC_BASE_URL when pointing at a proxy or non-Anthropic
# endpoint (e.g. for `glm-5`). Use ANTHROPIC_API_KEY alone for direct Anthropic auth.
SDK_ENV = {
    "ANTHROPIC_AUTH_TOKEN": os.getenv("ANTHROPIC_AUTH_TOKEN", ""),
    "ANTHROPIC_BASE_URL": os.getenv("ANTHROPIC_BASE_URL", ""),
    "ANTHROPIC_API_KEY": os.getenv("ANTHROPIC_API_KEY", ""),
    "API_TIMEOUT_MS": "3000000",
}

# Default model — change to a Claude model id (e.g. "claude-sonnet-4-6") if not using a proxy.
DEFAULT_MODEL = "glm-5"

---
## 2. Basic Chat

The simplest pattern: send a prompt, collect text blocks from `AssistantMessage`.

In [ ]:
async def basic_chat(user_message: str, model: str = DEFAULT_MODEL) -> str:
    text_parts: list[str] = []
    async for message in query(
        prompt=user_message,
        options=ClaudeAgentOptions(
            allowed_tools=[],
            model=model,
            env=SDK_ENV,
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    text_parts.append(block.text)
    return "\n".join(text_parts)


print(await basic_chat("Reply in one short sentence: what is a civil code?"))

---
## 3. Chat with System Prompt

In [ ]:
async def chat_with_system_prompt(system_prompt: str, user_message: str, model: str = DEFAULT_MODEL) -> str:
    text_parts: list[str] = []
    async for message in query(
        prompt=user_message,
        options=ClaudeAgentOptions(
            system_prompt=system_prompt,
            allowed_tools=[],
            model=model,
            env=SDK_ENV,
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    text_parts.append(block.text)
    return "\n".join(text_parts)


system = """You are an expert Egyptian-law researcher.
Answer concisely and cite article numbers when possible. Use bullet points where appropriate."""

print(await chat_with_system_prompt(
    system_prompt=system,
    user_message="In one paragraph, what areas does the Egyptian Civil Code typically cover?",
))

---
## 4. Streaming

`query()` is already an async generator — print as text blocks arrive. The terminating `ResultMessage` carries cost/duration metadata.

In [ ]:
async def chat_streaming(system_prompt: str, user_message: str, model: str = DEFAULT_MODEL) -> None:
    async for message in query(
        prompt=user_message,
        options=ClaudeAgentOptions(
            system_prompt=system_prompt,
            allowed_tools=[],
            model=model,
            env=SDK_ENV,
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text, end="", flush=True)
        elif isinstance(message, ResultMessage):
            cost = getattr(message, "total_cost_usd", None) or 0.0
            print(f"\n\n--- Done | Cost: ${cost:.4f} | Duration: {message.duration_ms}ms ---")


await chat_streaming(
    system_prompt="You are a concise legal writer.",
    user_message="Write a 3-sentence summary of why codified civil law differs from common law.",
)

---
## 5. Reusable `chat()` Helper

Mirrors `openai.chat.completions.create()` shape — returns content + cost + duration.

In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class ChatResponse:
    content: str
    model: str
    cost_usd: float
    duration_ms: int


async def chat(
    user_message: str,
    system_prompt: Optional[str] = None,
    model: str = DEFAULT_MODEL,
) -> ChatResponse:
    options = ClaudeAgentOptions(
        system_prompt=system_prompt,
        allowed_tools=[],
        model=model,
        env=SDK_ENV,
    )

    text_parts: list[str] = []
    result_model, cost, duration = "", 0.0, 0
    async for message in query(prompt=user_message, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    text_parts.append(block.text)
        elif isinstance(message, ResultMessage):
            result_model = getattr(message, "model", "unknown")
            cost = getattr(message, "total_cost_usd", None) or 0.0
            duration = getattr(message, "duration_ms", 0)

    return ChatResponse(
        content="\n".join(text_parts),
        model=result_model,
        cost_usd=cost,
        duration_ms=duration,
    )


response = await chat(
    system_prompt="You are a helpful legal assistant. Be brief.",
    user_message="Name two principles common to most civil-law systems.",
)
print(response.content)
print(f"\nModel: {response.model}")
print(f"Cost:  ${response.cost_usd:.4f}")
print(f"Time:  {response.duration_ms}ms")

---
## 6. LightRAG Wrapper — Claude Agent SDK as the LLM Backend

[LightRAG](https://github.com/HKUDS/LightRAG) needs an async `llm_model_func` with the signature:

```python
async def llm_model_func(prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs) -> str
```

We adapt `claude_agent_sdk.query()` to that shape. We also need an embedding function — the Agent SDK doesn't provide one, so we use **Qwen3-Embedding 0.6B via Ollama**.

### 6.1 LLM Wrapper

In [13]:
# Model used by LightRAG for entity/relation extraction during indexing AND for query-time generation.
# Indexing makes a lot of calls, so prefer a fast/cheap model here.
RAG_LLM_MODEL = DEFAULT_MODEL  # e.g. "glm-5" via proxy, or "claude-haiku-4-5-20251001" direct


async def claude_agent_complete(
    prompt,
    system_prompt=None,
    history_messages=None,
    keyword_extraction=False,
    **kwargs,
) -> str:
    """LightRAG-compatible LLM function powered by the Claude Agent SDK."""
    history_messages = history_messages or []

    # Flatten history into the prompt (Agent SDK's query() takes a single string).
    parts: list[str] = []
    for msg in history_messages:
        role = msg.get("role", "user").capitalize()
        parts.append(f"{role}: {msg.get('content', '')}")
    parts.append(str(prompt))
    full_prompt = "\n".join(parts)

    options = ClaudeAgentOptions(
        system_prompt=system_prompt,
        allowed_tools=[],
        model=RAG_LLM_MODEL,
        env=SDK_ENV,
    )

    text_parts: list[str] = []
    async for message in query(prompt=full_prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    text_parts.append(block.text)

    return "\n".join(text_parts)

### 6.2 Embedding Function (Ollama / Qwen3-Embedding 0.6B)

| Model | Ollama Tag | Dim | Max Tokens |
|---|---|---|---|
| **Qwen3 Embedding 0.6B** | `qwen3-embedding:0.6b` | 1024 | 32768 |
| Qwen3 Embedding 4B | `qwen3-embedding:4b` | 2560 | 32768 |
| Nomic Embed Text | `nomic-embed-text:latest` | 768 | 8192 |

In [14]:
import numpy as np
import httpx
from lightrag.utils import wrap_embedding_func_with_attrs

OLLAMA_BASE_URL = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_EMBED_MODEL = "qwen3-embedding:0.6b"
OLLAMA_EMBED_DIM = 1024
OLLAMA_EMBED_MAX_TOKENS = 32768


@wrap_embedding_func_with_attrs(
    embedding_dim=OLLAMA_EMBED_DIM,
    max_token_size=OLLAMA_EMBED_MAX_TOKENS,
    model_name=OLLAMA_EMBED_MODEL,
)
async def ollama_embedding_func(texts: list[str]) -> np.ndarray:
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": OLLAMA_EMBED_MODEL, "input": texts},
        )
        resp.raise_for_status()
        data = resp.json()
    return np.array(data["embeddings"], dtype=np.float32)

### 6.3 Sanity-check both wrappers

In [15]:
print("--- Testing Claude Agent SDK LLM ---")
print(await claude_agent_complete("What is a knowledge graph? Reply in one sentence."))

print("\n--- Testing Ollama embedding ---")
embeddings = await ollama_embedding_func(["civil code", "criminal procedure"])
print(f"Model: {OLLAMA_EMBED_MODEL}")
print(f"Shape: {embeddings.shape}")
sim = float(np.dot(embeddings[0], embeddings[1]) / (np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])))
print(f"Cosine similarity: {sim:.4f}")

INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude


--- Testing Claude Agent SDK LLM ---
A knowledge graph is a structured representation of real-world entities and their relationships, organized as a network of interconnected nodes (entities) and edges (relationships) that enables machines to reason about and navigate complex information.

--- Testing Ollama embedding ---
Model: qwen3-embedding:0.6b
Shape: (2, 1024)
Cosine similarity: 0.6919


---
## 7. Initialize LightRAG

Storage lives next to this notebook (inside the `agents/` folder) so the multi-agent system can point its RAG tool at the same `working_dir`.

In [12]:
from lightrag import LightRAG, QueryParam

AGENTS_DIR = PROJECT_ROOT / "src" / "legal_explainer" / "agents"
WORKING_DIR = AGENTS_DIR / "rag_storage_egyptian_law"
WORKING_DIR.mkdir(parents=True, exist_ok=True)

rag = LightRAG(
    working_dir=str(WORKING_DIR),
    llm_model_func=claude_agent_complete,
    llm_model_name=RAG_LLM_MODEL,
    embedding_func=ollama_embedding_func,

    # ── Concurrency ───────────────────────────────────────────────────────────
    # The Claude Agent SDK spawns a Node.js subprocess PER LLM call. 16 of those
    # simultaneously => "Command failed with exit code 1" from CLI death (memory
    # pressure or proxy burst-limit). 4 is a safe ceiling for the subprocess
    # transport; raise gradually if your machine + proxy can handle it.
    llm_model_max_async=8,
    # Ollama serves a single embedding model ~sequentially; >4 just queues.
    embedding_func_max_async=4,

    # ── Timeouts (the fix for "Worker timeout after 60s") ─────────────────────
    # Worker timeout = default_embedding_timeout * 2.
    # default_llm_timeout doubled => extraction worker ≤ 1200s, plenty for glm-5.
    default_embedding_timeout=180,
    default_llm_timeout=600,

    # ── Chunking ──────────────────────────────────────────────────────────────
    # Roughly one full page per chunk (~4000 tokens).
    chunk_token_size=4000,
    chunk_overlap_token_size=200,

    addon_params={"language": "English"},
)

await rag.initialize_storages()
print(f"LightRAG initialized. Storage: {WORKING_DIR}")

INFO: [] Created new empty graph file: /Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '/Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '/Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '/Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law/vdb_chunks.json'} 0 data


LightRAG initialized. Storage: /Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law


---
## 8. Extract text from `EgyptianLaw.pdf`

We use **PyMuPDF (`fitz`)** — it handles Arabic and mixed-script PDFs better than `pypdf` for most legal documents.

In [13]:
import fitz  # PyMuPDF

PDF_PATH = PROJECT_ROOT / "EgyptianLaw.pdf"
assert PDF_PATH.exists(), f"PDF not found at {PDF_PATH}"

doc = fitz.open(str(PDF_PATH))
pages_text: list[str] = []
for i, page in enumerate(doc):
    text = page.get_text("text")
    if text.strip():
        pages_text.append(text)
doc.close()

full_text = "\n\n".join(pages_text)
print(f"Pages extracted: {len(pages_text)}")
print(f"Total characters: {len(full_text):,}")
print("\n--- First 800 chars preview ---")
print(full_text[:800])

Pages extracted: 170
Total characters: 721,166

--- First 800 chars preview ---
 
القانون المدني المصري 
قانون اإلصدار 
 مادة
١ 
 يلغي القانون المدني المعمول به أمام المحاكم الوطنية والصادر في
٨٢ أكتوبر سنة 
٣٨٨١ 
 والقانون المدني المعمول به أمام المحاكم المختلطة والصادر في
٨٢ يونيو 
 سنة
٥٧٨١ 
 ويستعاض عنهما بالقانون المدني المرافق لهذا
القانون 
 مادة
٢ 
 على وزير العدل تنفيذ هذا القانون ويعمل به ابتداء من
١٥ أكتوبر سنة 
.٩٤٩١ 
نأمر بأن يبصم هذا القانون بخاتم الدولة وأن ينشر في الجريدة الرسمية وينفذ 
.كقانون من قوانين الدولة 
 صدر بقصر القبة في
٩ 
 رمضان سنة
٧٦٣١  ٦١ 
 يوليو سنة
٨٤٩١ 
نصوص القانون المد
نى 
باب تمهيدي 
أحكام عامة 
 
 الفصل األول 
القانون وتطبيقه 
SECTION I 
Laws and their Applications
١ -القانون والحق 
1. Laws and Rights
 ( مادة
١( 
 
)١ (
 تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه
النصوص في لفظها أو في فحواه.ا 
 
(٢ )
 فإذا لم يوجد نص 


---
## 9. Ingest into LightRAG

LightRAG handles chunking, entity/relation extraction, and embedding internally. Expect this to take **several minutes** for a full law document — each chunk triggers an LLM call for extraction.

In [ ]:
print(f"Inserting {len(full_text):,} chars from EgyptianLaw.pdf ...")
await rag.ainsert(full_text, ids=["egyptian_law"], file_paths=[str(PDF_PATH)])
print("Insert complete.")

INFO: Created 1 duplicate document records with track_id: insert_20260523_030839_1f6d3d01
INFO: Preserving 1 failed document entries for manual review
INFO: Reset 1 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: /Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/EgyptianLaw.pdf
INFO: Processing d-id: egyptian_law


Inserting 721,166 chars from EgyptianLaw.pdf ...


INFO: Embedding func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO: LLM func: 8 new workers initialized (Timeouts: Func: 600s, Worker: 1200s, Health Check: 1215s)
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO: Chunk 1 of 53 extracted 33 Ent + 28 Rel chunk-51653f52168ac08cb3725623b6116442
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/

Insert complete.


INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude
INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Volumes/Work/conda/envs/dev/lib/python3.11/site-packages/claude_agent_sdk/_bundled/claude


---
## 10. Query the Egyptian Law Graph

Three modes worth knowing:
- `naive` — pure vector search, fastest, good sanity check
- `hybrid` — entity-level + relationship-level retrieval, best quality
- `only_need_context=True` — returns retrieved chunks/entities/relations without an LLM call (useful when this RAG is a tool in a larger agent loop and the caller wants to do its own synthesis)

In [15]:
print("=== Naive Mode ===")
result = await rag.aquery(
    "What does Egyptian law say about contracts?",
    param=QueryParam(mode="naive"),
)
print(result)

INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 6 chunks


=== Naive Mode ===


INFO:  == LLM cache == saving: naive:query:bb0c5ecd03b5e276b694cd2151a65e4e


# Egyptian Law on Contracts

Based on the Egyptian Civil Code (القانون المدني المصري), contracts are governed by comprehensive rules covering their formation, validity, effects, and various types. Below is a detailed overview:

---

## 1. Formation of Contracts (Consent / الرضاء)

A **contract is formed** when two parties exchange matching expressions of intention, subject to any special formalities required by law (Article 89) [1].

### Expression of Intention
- An intention may be declared **verbally, in writing, by commonly used signs**, or by conduct that leaves no doubt as to its meaning (Article 90) [1].
- A declaration becomes effective when it reaches the person to whom it was addressed (Article 91) [1].

### Offer and Acceptance
- If a **time limit** is fixed for acceptance, the offeror is bound to maintain the offer until the expiration of that time limit (Article 93) [1].
- If an offer is made **during a meeting** (in person or by telephone) without a time limit, the offeror

In [16]:
print("=== Hybrid Mode ===")
result = await rag.aquery(
    "Summarise the rules around personal status and family law as covered in this document.",
    param=QueryParam(mode="hybrid"),
)
print(result)

=== Hybrid Mode ===


INFO:  == LLM cache == saving: hybrid:keywords:0c4cc9a21848f5b32efadddeb289fb6e
INFO: Query nodes: Marriage, Divorce, Inheritance, Guardianship, Custody, Legal personal status (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 144 relations
INFO: Query edges: Personal status, Family law, Legal rules summary, Document analysis (top_k:40, cosine:0.2)
INFO: Global query: 66 entites, 40 relations
INFO: Raw search results: 95 entities, 174 relations, 0 vector chunks
INFO: After truncation: 63 entities, 148 relations
INFO: Selecting 39 from 39 entity-related chunks by vector similarity
INFO: Find 4 additional chunks in 4 relations (deduplicated 34)
INFO: Selecting 4 from 4 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 43 -> 43 (deduplicated 0)
INFO: Final context: 63 entities, 148 relations, 3 chunks
INFO: Final chunks S+F/O: E8/1 R1/1 E4/2
INFO:  == LLM cache == saving: hybrid:query:8597bebabd594e1ffbe2aa7a1f951226


# Personal Status and Family Law in the Egyptian Civil Code

The Egyptian Civil Code addresses several key areas of personal status and family law, spanning legal personality, capacity, marriage, divorce, inheritance, guardianship, and alimony. Below is a comprehensive summary of the rules as found in the provided context.

---

## 1. Legal Personality and Capacity

**Legal personality** commences from the moment a child is **born alive** and ends at **death**. The law also determines the rights of an unborn child (*en ventre de sa mère*), as established in **Article 29** [1].

Birth and death are established through **official registers** kept for that purpose. Where such registers are unavailable or inaccurate, proof may be established by any other means (**Article 30**) [1].

### Degrees of Legal Capacity

The Code establishes a tiered system of legal capacity:

- **Full legal capacity** is attained at **twenty-one years of age** (Gregorian calendar), provided the person is in posse

In [17]:
# Retrieval-only mode — returns context without invoking the LLM. This is the shape
# a tool-wrapped version of this RAG would expose to a coordinator agent.
print("=== Retrieved Context Only (no LLM call) ===\n")
context = await rag.aquery(
    "What are the obligations of a tenant under Egyptian civil law?",
    param=QueryParam(mode="hybrid", only_need_context=True),
)
print(context)

=== Retrieved Context Only (no LLM call) ===



INFO:  == LLM cache == saving: hybrid:keywords:1913bf80ff203447bdde783605c25177
INFO: Query nodes: Tenant, Egyptian Civil Code, Lease agreement, Rent payment, Property maintenance, Lease contract (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 116 relations
INFO: Query edges: Tenant obligations, Egyptian civil law, Landlord-tenant law (top_k:40, cosine:0.2)
INFO: Global query: 50 entites, 40 relations
INFO: Raw search results: 73 entities, 123 relations, 0 vector chunks
INFO: After truncation: 66 entities, 123 relations
INFO: Selecting 40 from 40 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 123 relations
INFO: Round-robin merged chunks: 40 -> 40 (deduplicated 0)
INFO: Final context: 66 entities, 123 relations, 3 chunks
INFO: Final chunks S+F/O: E20/1 E8/2 E7/3



Knowledge Graph Data (Entity):

```json
{"entity": "Egyptian Civil Code", "type": "content", "description": "The Egyptian Civil Code (القانون المدني المصري) is a comprehensive legislative document governing civil matters in Egypt, including laws, rights, legal capacity, conflicts of law, and various civil obligations.<SEP>The body of civil law containing provisions governing suretyship relationships and real property rights in Egypt, as reflected in Articles 788–814."}
{"entity": "Contractual Obligations", "type": "concept", "description": "Contractual Obligations (الالتزامات التعاقدية) are governed by the law of the common domicile of the contracting parties, or by the law of the place where the contract was concluded."}
{"entity": "Chapter II: Contracts Relating to the Use of a Thing", "type": "content", "description": "A chapter of the civil code that covers contracts for the use of a thing, including leases generally."}
{"entity": "Egypt", "type": "location", "description": "Egypt

---
## 11. Cleanup

Always finalize storages so KV stores and the graph get flushed to disk.

In [18]:
await rag.finalize_storages()
print(f"Storages finalized. Persisted graph + vector dbs live at: {WORKING_DIR}")

INFO: Successfully finalized 12 storages


Storages finalized. Persisted graph + vector dbs live at: /Volumes/Shared/NileUni/GenAI/LegalPolicy_LLM/src/legal_explainer/agents/rag_storage_egyptian_law


---
## 12. Next Step — Exposing this as an Agent Tool

In the multi-agent system, this RAG will be wrapped as a tool the coordinator can call. Sketch:

```python
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

@tool(
    "search_egyptian_law",
    "Search the Egyptian law knowledge graph and return relevant context.",
    {"question": str, "mode": str},  # mode: "naive" | "local" | "global" | "hybrid"
)
async def search_egyptian_law(args):
    # Reuse the same WORKING_DIR initialized above.
    rag_local = LightRAG(
        working_dir=str(WORKING_DIR),
        llm_model_func=claude_agent_complete,
        llm_model_name=RAG_LLM_MODEL,
        embedding_func=ollama_embedding_func,
    )
    await rag_local.initialize_storages()
    context = await rag_local.aquery(
        args["question"],
        param=QueryParam(mode=args.get("mode", "hybrid"), only_need_context=True),
    )
    await rag_local.finalize_storages()
    return {"content": [{"type": "text", "text": context}]}

legal_rag_server = create_sdk_mcp_server(
    name="legal_rag",
    version="0.1.0",
    tools=[search_egyptian_law],
)

options = ClaudeAgentOptions(
    mcp_servers={"legal_rag": legal_rag_server},
    allowed_tools=["mcp__legal_rag__search_egyptian_law"],
    env=SDK_ENV,
    model=DEFAULT_MODEL,
)
```

Returning `only_need_context=True` keeps the tool cheap and gives the coordinator agent the freedom to synthesise across multiple tool calls.